# V95 - Ordinal regression

Pre-registered plan: relevance is graded and discrete, so cumulative-link ordinal regression may fit the level structure better than plain regression. LogisticAT per module on word-only features under grouped folds with a fresh seed, gated against Ridge with the usual locks.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from mord import LogisticAT
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 2020
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v95")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Ordinal vs Ridge heads, same folds

Labels quantize to integer grades for the ordinal head. Ranking uses expected grade.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in list(train["user_id"])]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
levels = np.unique(Y)
YQ = np.searchsorted(levels, Y)
print(f"discrete levels: {len(levels)}")

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

oof_r, oof_o, fids = np.zeros_like(Y), np.zeros_like(Y), np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(W, strata, fold_groups)):
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(W[tr], Y[tr, j])
        oof_r[va, j] = m.predict(W[va])
        vals, inv = np.unique(Y[tr, j], return_inverse=True)
        o = LogisticAT(alpha=1.0)
        o.fit(W[tr], inv)
        probs = o.predict_proba(W[va])
        oof_o[va, j] = (probs * vals).sum(1)
    fids[va] = fi + 1
base = ndcg5(Y, oof_r)
d = ndcg5(Y, oof_o) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fids == k], oof_o[fids == k]).mean() - base[fids == k].mean() for k in range(1, N_SPLITS + 1))
print(f"ridge={base.mean():.5f} ordinal={ndcg5(Y, oof_o).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT ordinal" if (lo > GATE_FLOOR and wf >= 0) else "KEEP Ridge; ordinal structure adds nothing")

discrete levels: 57


ridge=0.62614 ordinal=0.61546 delta=-0.01068 CI95=[-0.01317,-0.00814] worst-fold=-0.03157
KEEP Ridge; ordinal structure adds nothing
